# Step 1a: Load and Filter arXiv q-bio Papers

## Goal
Read the raw arXiv metadata snapshot and extract only papers
whose `categories` field contains at least one `q-bio.*` subcategory.
Clean the text fields and normalize metadata into a consistent schema.

## Input
- `/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json`
- Format: one JSON object per line (JSONL), ~2.5M records total

## Processing Steps

### 1. Stream-read the file line by line
The snapshot is too large to load all at once into memory.
We read it line-by-line and filter on the fly.

### 2. Filter by category
The `categories` field is a space-separated string (e.g., `"nlin.PS q-bio.MN"`).
We split on whitespace and keep a record only if at least one token starts with `q-bio`.
This is more precise than a plain substring match.

### 3. Clean title and abstract
- Strip leading/trailing whitespace
- Replace internal newline characters (`\n`) with a single space
- This ensures each title/abstract is a single clean line

### 4. Parse authors from `authors_parsed`
`authors_parsed` is a list of `[last, first, suffix]` sublists.
We join them as `"Last, First"` (dropping empty suffix).
Example: `[["Smith", "John", ""], ["Lee", "Amy", ""]]`
→ `["Smith, John", "Lee, Amy"]`

### 5. Extract submission year
Primary source: `versions[0]['created']`  
→ Parse the date string (e.g., `"Mon, 3 Apr 2006 00:00:00 GMT"`) and extract the 4-digit year.  
Fallback: if `versions` is missing or malformed, use `update_date[:4]`.

### 6. Output schema
Each record contains exactly these 6 fields:

| Field      | Source                           | Type      |
|------------|----------------------------------|-----------|
| paper_id   | `id`                             | str       |
| title      | `title` (cleaned)                | str       |
| abstract   | `abstract` (cleaned)             | str       |
| authors    | `authors_parsed` → "Last, First" | list[str] |
| categories | `categories`                     | str       |
| year       | `versions[0]['created']` (year)  | str (4-digit) |

**Note:** `institution` and `region` fields are NOT added here.
Those will be enriched in a later step via the OpenAlex API.

## Output
- `/kaggle/working/qbio_papers.json`
- Format: JSON array of records (one dict per paper)

## Validation
- Total record count
- First 3 records (spot-check)
- Output file size on disk

## Next Step
After confirming the output, publish `qbio_papers.json` as a Kaggle Dataset,
then proceed to Step 2: generating embeddings with `all-MiniLM-L6-v2`.

In [1]:
import json
import os

# ── Input / Output paths ──────────────────────────────────────────────────────
INPUT_PATH = "/kaggle/input/datasets/organizations/Cornell-University/arxiv/arxiv-metadata-oai-snapshot.json"
OUTPUT_PATH = "/kaggle/working/qbio_papers.json"

# ── Helper: parse submission year from versions[0]['created'] ─────────────────
def extract_year(record):
    """
    Try to extract the 4-digit submission year from versions[0]['created'].
    Example date string: 'Mon, 3 Apr 2006 00:00:00 GMT'
    Falls back to update_date[:4] if versions field is missing or unparseable.
    """
    try:
        created_str = record["versions"][0]["created"]
        year = created_str.strip().split()[3]
        if len(year) == 4 and year.isdigit():
            return year
    except (KeyError, IndexError, AttributeError):
        pass
    try:
        return record["update_date"][:4]
    except (KeyError, TypeError):
        return "unknown"

# ── Helper: build "Last, First" author list from authors_parsed ───────────────
def parse_authors(record):
    """
    authors_parsed is a list of [last, first, suffix] sublists.
    We format each as "Last, First", dropping empty parts.
    Returns a list of strings.
    """
    authors_parsed = record.get("authors_parsed", [])
    result = []
    for parts in authors_parsed:
        last  = parts[0].strip() if len(parts) > 0 else ""
        first = parts[1].strip() if len(parts) > 1 else ""
        if last and first:
            result.append(f"{last}, {first}")
        elif last:
            result.append(last)
    return result

# ── Helper: clean whitespace / newlines from a text field ────────────────────
def clean_text(text):
    """
    Split on any whitespace (including \n \t and multiple spaces),
    then rejoin with a single space. More thorough than replace('\n', ' ').
    """
    if not text:
        return ""
    return " ".join(text.split())

# ── Main: stream-read snapshot, filter q-bio, collect records ─────────────────
qbio_papers = []

print("Reading snapshot — this may take several minutes...")

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            continue  # skip malformed lines

        # Filter: keep only papers where at least one category starts with "q-bio"
        categories = record.get("categories", "")
        if not any(cat.startswith("q-bio") for cat in categories.split()):
            continue

        paper = {
            "paper_id":   record.get("id", ""),
            "title":      clean_text(record.get("title", "")),
            "abstract":   clean_text(record.get("abstract", "")),
            "authors":    parse_authors(record),
            "categories": categories,
            "year":       extract_year(record),
        }
        qbio_papers.append(paper)

        # Progress indicator every 100k lines
        if (line_num + 1) % 100_000 == 0:
            print(f"  Scanned {line_num + 1:,} lines | q-bio collected: {len(qbio_papers):,}")

print(f"\nDone. Total q-bio papers collected: {len(qbio_papers):,}")

# ── Write output ──────────────────────────────────────────────────────────────
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(qbio_papers, f, ensure_ascii=False, indent=2)

# ── Validation ────────────────────────────────────────────────────────────────
file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)

print(f"\n=== Validation ===")
print(f"Total records : {len(qbio_papers):,}")
print(f"Output file   : {OUTPUT_PATH}")
print(f"File size     : {file_size_mb:.2f} MB")
print(f"\nFirst 3 records (spot-check):")
for i, paper in enumerate(qbio_papers[:3]):
    print(f"\n--- Record {i+1} ---")
    print(f"  paper_id  : {paper['paper_id']}")
    print(f"  title     : {paper['title'][:80]}...")
    print(f"  abstract  : {paper['abstract'][:100]}...")
    print(f"  authors   : {paper['authors'][:3]}")
    print(f"  categories: {paper['categories']}")
    print(f"  year      : {paper['year']}")

Reading snapshot — this may take several minutes...

Done. Total q-bio papers collected: 55,301

=== Validation ===
Total records : 55,301
Output file   : /kaggle/working/qbio_papers.json
File size     : 81.56 MB

First 3 records (spot-check):

--- Record 1 ---
  paper_id  : 0704.0021
  title     : Molecular Synchronization Waves in Arrays of Allosterically Regulated Enzymes...
  abstract  : Spatiotemporal pattern formation in a product-activated enzymic reaction at high enzyme concentratio...
  authors   : ['Casagrande, Vanessa', 'Togashi, Yuichi', 'Mikhailov, Alexander S.']
  categories: nlin.PS physics.chem-ph q-bio.MN
  year      : 2007

--- Record 2 ---
  paper_id  : 0704.0034
  title     : Origin of adaptive mutants: a quantum measurement?...
  abstract  : This is a supplement to the paper arXiv:q-bio/0701050, containing the text of correspondence sent to...
  authors   : ['Ogryzko, Vasily']
  categories: q-bio.PE q-bio.CB quant-ph
  year      : 2007

--- Record 3 ---
  paper_id 